In [1]:
import os
import rasterio
import numpy as np
from pathlib import Path
import pandas as pd
import re 
from metloom.pointdata import SnotelPointData
import pandas as pd
from datetime import datetime



In [3]:
# Path to the folder containing raster files
path = "C:/Users/RDCRLSMC/Desktop/snowdepth/"
files = os.listdir(path)
# sortList = 0, 3, 4, 1, 2
# files = files[sortList]
print(files)

['20240315_MCS-snowdepth.tif', '637_25_YEAR=2022.csv', '637_25_YEAR=2023.csv', '637_25_YEAR=2024.csv', 'full.csv', 'MCS_20220217_SNOWDEPTH.tif', 'MCS_20220317_SNOWDEPTH.tif', 'MCS_20220407_SNOWDEPTH.tif', 'MCS_20230405_SNOWDEPTH.tif', 'MCS_20231113_SNOWDEPTH.tif', 'MCS_20231228_SNOWDEPTH.tif', 'MCS_20240115_SNOWDEPTH.tif', 'MCS_20240213_SNOWDEPTH.tif', 'MCS_20240315_SNOWDEPTH.tif', 'MCS_20240418_SNOWDEPTH.tif', 'notworking', 'snow_depths.csv', 'snow_depths_edited.csv', 'test', 'testsnow_depths.csv']


In [73]:
from pathlib import Path
import rasterio
import numpy as np
import re

def calculate_raster_medians(folder_path):
    # Find all .tif files in the folder
    raster_files = Path(folder_path).glob('*.tif')

    # Dictionary to store median values for each raster
    raster_medians = {}

    for raster_file in raster_files:
        # Extract integers from the filename to use as the key
        numeric_part = re.search(r'\d+', raster_file.name)
        if numeric_part:  # Ensure there is a numeric part
            key = int(numeric_part.group(0))  # Convert to an integer
            
            
            with rasterio.open(raster_file) as src:
                # Read the data as a 2D numpy array (assuming single band rasters)
                data = src.read(1)

                # Compute median and store
                median_value = np.nanmedian(data)
                raster_medians[key] = median_value
                
                # Convert dictionary to DataFrame
                df = pd.DataFrame(rastermedians.items(), columns=['File Name', 'Median Snow Depth (m)'])

                # Export to CSV
                csv_file_path = folder_path + 'snow_depths.csv'
                df.to_csv(csv_file_path, index=False)

    return raster_medians



In [74]:
rastermedians = calculate_raster_medians(path)
print(rastermedians)

PermissionError: [Errno 13] Permission denied: 'C:/Users/RDCRLSMC/Desktop/snowdepth/snow_depths.csv'

In [108]:
df1 = pd.read_csv(path + 'snow_depths.csv')
#print(df1['File Name'])
df1['Date'] = pd.to_datetime(df1['File Name'], format="%Y%m%d")
#df1['Date'] = df1['Date'].dt.strftime("%Y, %m, %d")
df1['Formatted_Date'] = df1['Date'].dt.strftime("%Y, %#m, %#d")


print(df1)

   File Name  Median Snow Depth       Date Formatted_Date
0   20240315           1.902100 2024-03-15    2024, 3, 15
1   20220217           1.326172 2022-02-17    2022, 2, 17
2   20220317           1.183472 2022-03-17    2022, 3, 17
3   20220407           0.920776 2022-04-07     2022, 4, 7
4   20230405           2.778564 2023-04-05     2023, 4, 5
5   20231113           0.244141 2023-11-13   2023, 11, 13
6   20231228           0.649902 2023-12-28   2023, 12, 28
7   20240115           1.154907 2024-01-15    2024, 1, 15
8   20240213           0.996094 2024-02-13    2024, 2, 13
9   20240418           1.283935 2024-04-18    2024, 4, 18


In [144]:
snotel_point = SnotelPointData("637:ID:SNTL", "MCS") # define the snotel ID

# Initialize list to collect data
all_data = []
df1['Snotel SDepth (in)'] = None  # Initialize column in df1

# Loop through the formatted dates and fetch data
for i, formatted_date in enumerate(df1['Formatted_Date']):
    try:
        # Split the formatted string into components to convert to a datetime object
        year, month, day = map(int, formatted_date.split(", "))
        start_date = datetime(year, month, day)

        # Fetch daily data
        data = snotel_point.get_daily_data(
            start_date, start_date,  # Pass datetime object as start and end date
            [snotel_point.ALLOWED_VARIABLES.SNOWDEPTH]
        )

        # Append the data to the list
        #all_data.append(data)

        # Extract the SNOWDEPTH value and assign it to the corresponding row in df1
        snow_depth = (
            data['SNOWDEPTH'] if isinstance(data, dict)
            else data['SNOWDEPTH'].values[0]
        )
        df1.loc[i, 'Snotel SDepth (in)'] = snow_depth
    
    except Exception as e:
        print(f"Failed to fetch data for {formatted_date}: {e}")
    

df1["Snotel SDepth (m)"] = df1["Snotel SDepth (in)"] * 0.0254 # convert snow depth from inches to meters

# Display the updated df1
print(df1)


   File Name  Median Snow Depth       Date Formatted_Date Snotel SDepth (in)  \
0   20240315          1.9020996 2024-03-15    2024, 3, 15               77.0   
1   20220217          1.3261719 2022-02-17    2022, 2, 17               55.0   
2   20220317          1.1834717 2022-03-17    2022, 3, 17               53.0   
3   20220407         0.92077637 2022-04-07     2022, 4, 7               39.0   
4   20230405          2.7785645 2023-04-05     2023, 4, 5              107.0   
5   20231113         0.24414062 2023-11-13   2023, 11, 13                1.0   
6   20231228         0.64990234 2023-12-28   2023, 12, 28               19.0   
7   20240115          1.1549072 2024-01-15    2024, 1, 15               46.0   
8   20240213         0.99609375 2024-02-13    2024, 2, 13               51.0   
9   20240418          1.2839355 2024-04-18    2024, 4, 18               54.0   

   Snotel SDepth (m)  
0             1.9558  
1              1.397  
2 1.3461999999999998  
3 0.9905999999999999  
4   

In [145]:
df1.to_csv("C:/Users/RDCRLSMC/Desktop/snowdepth/full.csv", index=False)